# Module 01: Foundations — Building Your First Agent

## What is an AI Agent?

An AI agent is **not just a chatbot**. A chatbot takes a single prompt and returns a single response — one shot, done. An agent is different: it can **take actions**, observe what happened, and keep going until the task is complete.

The mechanism that makes this possible is the **agent loop**:

```
┌─────────────────────────────────────────────────┐
│                  AGENT LOOP                     │
│                                                 │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐ │
│   │  THINK   │───▶│   ACT    │───▶│ OBSERVE  │ │
│   │(LLM rea- │    │(run code │    │(get tool │ │
│   │  sons)   │◀───│ or tool) │    │ result)  │ │
│   └──────────┘    └──────────┘    └──────────┘ │
│         ▲                               │       │
│         └───────── repeat ─────────────┘       │
└─────────────────────────────────────────────────┘
```

**THINK:** The LLM reads the full conversation history (task + all previous steps) and decides what to do next — either run some code, call a tool, or produce a final answer.

**ACT:** The framework executes whatever the LLM decided: runs a Python code block, calls a tool function, or records the final answer.

**OBSERVE:** The result of the action (stdout, return value, error, or tool output) is captured and appended to the history.

The loop then repeats — the LLM sees the updated history including the new observation and reasons about the next step.

**When does it stop?**
- The agent calls `final_answer()` — this is the explicit signal that the task is done. The argument becomes the return value of `agent.run()`.
- The agent hits `max_steps` (default: 20) — a safety limit to prevent infinite loops.

This loop is the same regardless of task complexity. smolagents manages the loop for you — your job is to provide the model, the tools, and the task.

## Setup

In [ ]:
# Install required packages
# Uncomment the line below if running in Google Colab or a fresh environment
# !uv pip install smolagents python-dotenv duckduckgo-search mlflow
# Or using pip:
# !pip install smolagents python-dotenv duckduckgo-search mlflow

In [ ]:
import os

# ----- HF_TOKEN Setup -----
# Option A: Load from .env file (local development)
# from dotenv import load_dotenv
# load_dotenv()

# Option B: Google Colab Secrets
# Uncomment the lines below when running in Google Colab.
# Go to: Colab → Secrets (🔑 icon) → Add HF_TOKEN
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Option C: Set directly (not recommended for shared notebooks)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

In [ ]:
import os
from dotenv import load_dotenv
from smolagents import CodeAgent, InferenceClientModel

load_dotenv()

# Free-tier HuggingFace model via Inference API
# Requires HF_TOKEN in your .env file
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=os.environ["HF_TOKEN"],
)

print("Model initialized:", model.model_id)

## Your First Agent

We create a `CodeAgent` with `add_base_tools=True` which gives it a Python interpreter. It writes Python code to solve problems — watch the output to see it think step by step.

In [ ]:
agent = CodeAgent(tools=[], model=model, add_base_tools=True)
result = agent.run("What is the 15th Fibonacci number? Show your work.")
print("\nFinal answer:", result)

## Inspecting Agent Steps

Every step the agent took is stored in `agent.memory.steps`. This is invaluable for debugging and understanding agent behavior.

In [ ]:
print(f"Total steps: {len(agent.memory.steps)}\n")
for i, step in enumerate(agent.memory.steps):
    step_type = type(step).__name__
    print(f"Step {i}: [{step_type}]")
    print(str(step)[:300])
    print("---")

## Guided Example: Multi-Step Reasoning

Now let's give the agent a slightly more complex task that requires multiple steps.

In [ ]:
result = agent.run(
    "Create a list of the first 10 prime numbers, then calculate their sum and average."
)
print("\nFinal answer:", result)
print(f"Steps used: {len(agent.memory.steps)}")

## Exercises

Complete the cells below. Each has a `# TODO` comment guiding you.

In [ ]:
# TODO Exercise 1: Temperature conversion agent
# Create a new CodeAgent (same model, add_base_tools=True)
# Ask it: "Convert 98.6°F to Celsius and Kelvin. Show the formula used."
# After it runs, print: the result AND how many steps it took.

# Your code here:

In [ ]:
# TODO Exercise 2: Data task
# Ask the agent: "Given the list [4, 7, 2, 9, 1, 5], find the median
# without importing the statistics library. Show each step."
# Inspect the steps afterward — what type of steps did it take?

# Your code here:

## What You Built

You now know how to:
- Understand the agent loop: Think → Act → Observe
- Initialize `InferenceClientModel` with a free HF model
- Create and run a `CodeAgent`
- Inspect `agent.memory.steps` to understand agent reasoning

**Key insight:** The agent loop is the same regardless of the task complexity. smolagents handles the loop — you just provide the model, tools, and task.

## Next Module Preview

**Module 02: Tools & Custom Tools**

Right now your agent can only use built-in base tools. In Module 02 you'll learn how to build your own tools — giving your agent custom capabilities like fetching data from APIs, analyzing files, or running domain-specific logic.